# 베드와 출력물이 함께 보이는 이미지 GAN 예비 생성

기존 언더필 촬영에서 베드와 출력물이 함께 보이는 **실제 사진 50장**을 골라, 입문용 DCGAN으로 **256×256 이미지 50장**을 생성합니다. 이번에는 분류기를 학습하거나 성능을 비교하지 않습니다.

원본 1600×1200에서 기존 1000×1000 고정 영역을 자르고 256×256으로 줄입니다. 사진을 회색으로 마스킹하지 않아 베드도 모델 입력에 그대로 포함됩니다. 기존 자동 마스크의 표면 추정 비율 25~80%는 두 영역이 함께 보이도록 정한 **임시 선정값**이지 정답 판정이 아닙니다. 출력 조건 라벨만으로 모든 사진에서 결함이 뚜렷하다고 보장할 수 없습니다.

기존 사진의 fold 0 및 fold 4 평가 구간과 30프레임 완충 구간을 제외한 학습 구간에서만 50장을 선택합니다. 추후 분류기 실험의 평가 데이터가 GAN 학습에 섞이지 않도록 하기 위함입니다. 한 번의 출력에 속하는 연속 촬영 이미지이므로 독립적인 출력 50회는 아닙니다.

**사용 전**: 런타임 유형에서 **G4 GPU**를 선택합니다. 필요한 ZIP 3개는 Drive `내 드라이브/산공종설/`에 있습니다. 아래 셀 하나를 실행하면 입력 준비→GAN 학습→50장 생성→검토판·결과 ZIP을 Drive에 저장하고 자동으로 런타임을 반납합니다. 오류가 난 경우에도 오류를 기록하려 시도한 뒤 런타임을 반납합니다. GPU 사용량이 발생합니다.

1,500 업데이트·50장은 비교를 위한 이전과 같은 예비 설정으로, 최적값이라 확정된 것은 아닙니다. 생성 이미지에는 비현실적 패턴이나 원본 베드 반짝임의 모사가 생길 수 있어 결과를 직접 확인해야 합니다.


In [ ]:
from google.colab import runtime, drive
import traceback, sys, os, json, hashlib, time, subprocess, shutil, tempfile, zipfile
from pathlib import Path

MOUNTED = False
RUN_ROOT = None
status = {'status': 'starting'}
try:
    drive.mount('/content/drive')
    MOUNTED = True
    import numpy as np
    import torch
    from PIL import Image, ImageOps, ImageDraw
    from IPython.display import display

    DRIVE_DIR = Path('/content/drive/MyDrive/산공종설')
    ORIGINAL = DRIVE_DIR / 'underfill_originals_4000.zip'
    GRID = DRIVE_DIR / 'dinov2_grid_bundle_20261013.zip'
    CODE = DRIVE_DIR / 'gan_bed_pilot_code_20261101.zip'
    RUN_NAME = 'gan_bed_20261101_run01'
    SEED, STEPS, BATCH = 42, 1500, 8
    RUN_ROOT = DRIVE_DIR / 'results' / RUN_NAME
    PREPARED, GAN = RUN_ROOT/'prepared', RUN_ROOT/'gan'
    PREVIEW, LOGS = RUN_ROOT/'review', RUN_ROOT/'logs'
    CODE_DIR, CACHE = Path('/content/gan_bed_code'), Path('/content/gan_bed_inputs')
    for item in [ORIGINAL, GRID, CODE]:
        if not item.is_file(): raise FileNotFoundError(f'필요한 파일 없음: {item}')
    if not torch.cuda.is_available(): raise RuntimeError('GPU가 없습니다. 런타임 유형을 G4 GPU로 선택하세요.')
    gpu = torch.cuda.get_device_name(0)
    print('사용 GPU:',gpu,flush=True)
    if 'RTX PRO 6000' not in gpu.upper():
        raise RuntimeError(f'요청한 G4(RTX PRO 6000)가 아닙니다: {gpu}. 런타임 유형을 G4로 설정해주세요.')
    print('결과 위치:',RUN_ROOT,flush=True)
    def sha(path):
        h=hashlib.sha256()
        with Path(path).open('rb') as f:
            for b in iter(lambda:f.read(4*1024*1024),b''): h.update(b)
        return h.hexdigest()
    if sha(CODE) != 'f619b581c33841d405b8355df3ea34362b77c5b56b928ad96eeb72c320bda927':
        raise ValueError('노트북과 코드 ZIP이 다른 버전입니다.')
    settings={'seed':SEED,'steps':STEPS,'batch':BATCH,'gpu':gpu,
              'original_sha256':sha(ORIGINAL),'grid_sha256':sha(GRID),'code_sha256':sha(CODE),
              'scope':'generation only; no classifier', 'image_count':50}
    RUN_ROOT.mkdir(parents=True,exist_ok=True)
    LOGS.mkdir(exist_ok=True); PREVIEW.mkdir(exist_ok=True)
    lock=RUN_ROOT/'run_settings.json'
    if lock.exists():
        old=json.loads(lock.read_text(encoding='utf-8'))
        for key in ['seed','steps','batch','original_sha256','grid_sha256','code_sha256']:
            if old[key]!=settings[key]: raise ValueError('기존 실험과 설정이 다릅니다. 새 RUN_NAME을 지정하세요.')
    else: lock.write_text(json.dumps(settings,ensure_ascii=False,indent=2),encoding='utf-8')
    CODE_DIR.mkdir(parents=True,exist_ok=True); CACHE.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(CODE) as z:
        required={'prepare_data.py','prepare_bed_data.py','gan_model.py'}
        if set(z.namelist())!=required: raise ValueError('코드 ZIP 내용이 예상과 다릅니다.')
        for name in required: (CODE_DIR/name).write_bytes(z.read(name))
    def cache(src, name, expected):
        dst=CACHE/(name+'_'+expected[:16]+'.zip')
        if dst.exists():
            if sha(dst)!=expected: raise ValueError('임시 입력 파일 해시 불일치')
            return dst
        fd,tmp=tempfile.mkstemp(prefix=name,suffix='.part',dir=CACHE); os.close(fd)
        try:
            shutil.copyfile(src,tmp)
            if sha(tmp)!=expected: raise ValueError('임시 복사본 해시 불일치')
            os.replace(tmp,dst)
        finally:
            if os.path.exists(tmp): os.unlink(tmp)
        return dst
    local_original=cache(ORIGINAL,'original',settings['original_sha256'])
    local_grid=cache(GRID,'grid',settings['grid_sha256'])
    def run_logged(cmd, filename):
        print('실행:', ' '.join(str(x) for x in cmd),flush=True)
        with (LOGS/filename).open('a',encoding='utf-8') as log:
            log.write('\n--- '+time.strftime('%Y-%m-%d %H:%M:%S')+' ---\n'); log.flush()
            proc=subprocess.Popen([str(x) for x in cmd],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                  text=True,encoding='utf-8',errors='replace',bufsize=1)
            try:
                for line in proc.stdout:
                    log.write(line); log.flush(); print(line,end='',flush=True)
                result=proc.wait()
            except BaseException:
                proc.terminate(); raise
            if result: raise RuntimeError(f'실행 실패({result}). 로그: {filename}')
    if (PREPARED/'selection_settings.json').exists():
        print('기존 완료된 사진 준비 결과 재사용',flush=True)
    else:
        if PREPARED.exists() and any(PREPARED.iterdir()):
            raise RuntimeError('이전 사진 준비가 중단됐습니다. 기존 파일은 보존하고 RUN_NAME을 바꿔주세요.')
        run_logged([sys.executable,CODE_DIR/'prepare_bed_data.py','--original-zip',local_original,
                    '--bundle',local_grid,'--out',PREPARED,'--seed',SEED],'prepare.log')
    data=PREPARED/'data'/'train_real'/'underfill'
    actual=sorted(data.glob('*.png'))
    if len(actual)!=50: raise RuntimeError(f'실사 입력 개수 {len(actual)}장; 50장이 필요합니다.')
    print('베드·표면이 포함된 학습 후보 50장 준비 완료',flush=True)
    run_logged([sys.executable,CODE_DIR/'gan_model.py','--data',data,'--out',GAN,'--steps',STEPS,
                '--seed',SEED,'--num-generate',50,'--device','cuda','--batch',BATCH,
                '--save-every',100,'--log-every',25],'gan.log')
    progress=json.loads((GAN/'progress.json').read_text(encoding='utf-8'))
    manifest=json.loads((GAN/'training_manifest.json').read_text(encoding='utf-8'))
    generated=sorted((GAN/'generated').glob('*.png'))
    if progress.get('status')!='completed' or progress.get('step')!=STEPS or len(generated)!=50:
        raise RuntimeError('GAN이 완료되지 않았거나 생성 이미지가 50장이 아닙니다.')
    if {r['filename']:r['sha256'] for r in manifest['images']}!={p.name:sha(p) for p in actual}:
        raise RuntimeError('GAN 학습 목록과 현재 실사 50장의 해시가 다릅니다.')
    def make_sheet(paths,dest):
        side=180; cap=24; cols=10; rows=(len(paths)+cols-1)//cols
        sheet=Image.new('RGB',(side*cols,(side+cap)*rows),'white'); draw=ImageDraw.Draw(sheet)
        for i,p in enumerate(paths):
            x=(i%cols)*side; y=(i//cols)*(side+cap)
            with Image.open(p) as img:
                tile=ImageOps.fit(img.convert('RGB'),(side,side),method=Image.Resampling.LANCZOS)
            sheet.paste(tile,(x,y)); draw.text((x+3,y+side+3),p.stem[:24],fill='black')
        sheet.save(dest,quality=92)
    make_sheet(generated,PREVIEW/'generated_bed_and_surface_50.jpg')
    # 단순 픽셀 유사도는 사람이 확인할 짝을 제시하며 복사 여부의 증거가 아니다.
    def thumb(p):
        with Image.open(p) as img: return np.asarray(img.convert('RGB').resize((64,64)),dtype=np.float32)/255
    real_arr=np.stack([thumb(p) for p in actual]); pairs=[]
    for p in generated:
        mse=((real_arr-thumb(p))**2).mean(axis=(1,2,3)); idx=int(mse.argmin())
        pairs.append((p,actual[idx],float(mse[idx])))
    import csv
    with (PREVIEW/'nearest_training_pairs.csv').open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.writer(f); w.writerow(['generated','nearest_real_training','pixel_mse_64'])
        w.writerows([(a.name,b.name,d) for a,b,d in pairs])
    for k in range(5):
        side=160; cap=24; sample=pairs[k*10:(k+1)*10]
        sheet=Image.new('RGB',(side*2,(side+cap)*len(sample)),'white'); d=ImageDraw.Draw(sheet)
        for i,(fake,real,error) in enumerate(sample):
            y=i*(side+cap)
            for j,p in enumerate((fake,real)):
                with Image.open(p) as img: sheet.paste(ImageOps.fit(img.convert('RGB'),(side,side)),(j*side,y))
            d.text((3,y+side+3),fake.stem,fill='black'); d.text((side+3,y+side+3),real.stem[:20],fill='black')
        sheet.save(PREVIEW/f'generated_vs_real_{k+1:02d}.jpg',quality=92)
    display(Image.open(PREVIEW/'generated_bed_and_surface_50.jpg').resize((900,510)))
    result=RUN_ROOT/(RUN_NAME+'_results.zip'); tmp=result.with_suffix('.zip.part')
    if result.exists():
        with zipfile.ZipFile(result) as z:
            if z.testzip() is not None: raise RuntimeError('기존 결과 ZIP이 손상됐습니다.')
        print('기존 검증된 결과 ZIP 유지:',result,flush=True)
    else:
        with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4) as z:
            for root in (PREPARED,GAN,PREVIEW,LOGS):
                for path in root.rglob('*'):
                    if path.is_file() and path.suffix.lower() not in {'.pt','.pth','.tmp'}:
                        z.write(path,path.relative_to(RUN_ROOT).as_posix())
            z.write(lock,lock.relative_to(RUN_ROOT).as_posix())
        with zipfile.ZipFile(tmp) as z:
            bad=z.testzip()
            if bad: raise RuntimeError(f'결과 ZIP 무결성 오류: {bad}')
        os.replace(tmp,result)
    status={'status':'completed','generated_count':50,'gpu':gpu,'result_zip':str(result),
            'warning':'시각적 확인과 품질 검증이 필요하며 25~80% 면적은 임시 기준'}
    (RUN_ROOT/'run_status.json').write_text(json.dumps(status,ensure_ascii=False,indent=2),encoding='utf-8')
    print('성공! 결과 ZIP:',result,flush=True)
except BaseException as exc:
    status={'status':'failed','error_type':type(exc).__name__,'error':str(exc),'traceback':traceback.format_exc()}
    print('실험 오류:',status['traceback'],flush=True)
    if RUN_ROOT is not None:
        try:
            RUN_ROOT.mkdir(parents=True,exist_ok=True)
            (RUN_ROOT/'run_status.json').write_text(json.dumps(status,ensure_ascii=False,indent=2),encoding='utf-8')
        except Exception as log_error:
            print('Drive에 오류 기록 실패:',log_error,flush=True)
finally:
    if MOUNTED:
        try:
            print('Drive 저장 내역을 동기화하고 마운트를 해제합니다.',flush=True)
            drive.flush_and_unmount()
        except BaseException as sync_error:
            print('Drive 동기화 확인 실패. Drive에서 파일을 별도 확인하세요:',sync_error,flush=True)
    print('G4 런타임 반납을 요청합니다.',flush=True)
    try: runtime.unassign()
    except BaseException as release_error:
        print('자동 반납 확인 실패. Colab에서 런타임 연결을 확인하세요:',release_error,flush=True)


## 결과 확인

실행 중 오류나 중단 여부를 `results/gan_bed_20261101_run01/run_status.json`에서 확인할 수 있습니다. `review/`에 생성 50장 확인판과 실사 비교판을 저장합니다. `generated_vs_real`의 왼쪽은 생성, 오른쪽은 단순 픽셀 차이가 가장 작은 학습 사진입니다. 단순 비교는 외관 점검용이며 복사 여부나 성능을 증명하지 않습니다.

마지막 `runtime.unassign()`은 현재 VM을 반납하므로 Colab 화면에 연결 종료가 나타날 수 있습니다. Drive에 저장된 결과는 유지됩니다. 재연결해 반복할 때는 같은 학습 설정을 사용하거나 `RUN_NAME`을 새 이름으로 바꾸세요.
